# 01. Data Cleaning & Healthcare Data Quality Audit
Hyderabad Community Clinic Network - 500,000+ Records Across 4 Hospitals

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime

df_opd = pd.read_csv('../data/raw/hyderabad_opd_visits.csv')
df_adm = pd.read_csv('../data/raw/hyderabad_admissions.csv')
df_lab = pd.read_csv('../data/raw/hyderabad_lab_tests.csv')
df_bill = pd.read_csv('../data/raw/hyderabad_billing.csv')

print(f"Total rows: {len(df_opd) + len(df_adm) + len(df_lab) + len(df_bill):,d}")

## 1. Age Standardisation (Parsing Date-of-Birth Entries)

In [ ]:
def parse_age(val, ref_year=2023):
    if pd.isna(val): return np.nan
    s = str(val).strip()
    iso = re.match(r'^(\d{4})[-/](\d{1,2})[-/](\d{1,2})$', s)
    if iso: return max(0, ref_year - int(iso.group(1)))
    dmy = re.match(r'^(\d{1,2})[-/](\d{1,2})[-/](\d{4})$', s)
    if dmy: return max(0, ref_year - int(dmy.group(3)))
    str_m = re.match(r'^(\d+)\s*(?:yrs|years|y)?$', s, re.IGNORECASE)
    if str_m: return int(str_m.group(1))
    try:
        f = float(s)
        return ref_year - int(f) if f > 1900 and f <= ref_year else int(f)
    except: return np.nan

df_opd['age_cleaned'] = df_opd['raw_age'].apply(parse_age)
df_opd['age_cleaned'] = df_opd['age_cleaned'].fillna(df_opd['age_cleaned'].median()).astype(int)
print("Age summary:", df_opd['age_cleaned'].describe())

## 2. Standardising Free-Text Diagnoses

In [ ]:
def categorize_diagnosis(text):
    if pd.isna(text): return 'General / Unspecified'
    t = str(text).lower()
    if any(k in t for k in ['diabet', 't2dm', 'dm-ii', 'dm2', 'sugar', 'neuropathy']): return 'Cardiometabolic (Diabetes)'
    elif any(k in t for k in ['hypertens', 'htn', 'cad', 'coronary', 'cardio', 'stent', 'dyslipid', 'cholesterol', 'fatty liver']): return 'Cardiovascular & Metabolic'
    elif any(k in t for k in ['asthma', 'astma', 'copd', 'bronch', 'urti', 'respirat', 'rhinitis']): return 'Respiratory'
    elif any(k in t for k in ['gastro', 'age ', 'vomit', 'diarrhea', 'pyrexia', 'fever', 'dengue', 'typhoid', 'malaria', 'uti']): return 'Acute Infectious / GI'
    elif any(k in t for k in ['osteoarth', 'oa ', 'knee', 'spondyl', 'backache', 'disc', 'sprain']): return 'Orthopedic & Musculoskeletal'
    elif any(k in t for k in ['antenatal', 'anc ', 'pregnan', 'pcod', 'pcos', 'pediatric', 'baby']): return 'Maternal & Child Health'
    elif any(k in t for k in ['executive', 'master health', 'screening', 'checkup', 'malaise']): return 'Preventive & Wellness'
    return 'General Medicine / Other'

df_opd['diagnosis_category'] = df_opd['diagnosis_raw'].apply(categorize_diagnosis)
print(df_opd['diagnosis_category'].value_counts())

## 3. Rectifying Double-Counted GST Billing Glitch

In [ ]:
ratio = df_bill['total_billed_amount'] / (df_bill['base_charge'] * 1.18)
double_gst_mask = ratio > 1.10
df_bill['net_billing_amount'] = np.where(double_gst_mask, np.round(df_bill['total_billed_amount'] / 1.18, 2), df_bill['total_billed_amount'])
print(f"Corrected invoices: {double_gst_mask.sum():,d} of {len(df_bill):,d}")